# Generación de los flujos de Mirai

Convierte las capturas originales del conjunto Mirai ([Mirai raw dataset](https://www.kaggle.com/datasets/nguyenquanitmo/mirai-raw-dataset)) en el fichero `data/flows.csv` que usan los cuadernos de Mirai. Cada flujo agrupa los paquetes de una misma conexión, identificada por las direcciones, los puertos y el protocolo, y toma la clase de la captura de la que procede.

Las capturas se descargan con `kagglehub.dataset_download("nguyenquanitmo/mirai-raw-dataset")`. Para pasarlas de pcapng a pcap hace falta `editcap`, que se instala con Wireshark.

In [1]:
import os
import shutil
import subprocess
import warnings
from collections import defaultdict
from pathlib import Path

import dpkt
import pandas as pd

warnings.filterwarnings('ignore')

EDITCAP = shutil.which('editcap') or r'C:\Program Files\Wireshark\editcap.exe'
DATA_PATH = Path.home() / '.cache' / 'kagglehub' / 'datasets' / 'nguyenquanitmo' / 'mirai-raw-dataset' / 'versions' / '2'
TEMP_DIR = Path.cwd().parent / 'data' / 'temp_pcap'
OUT_CSV = Path.cwd().parent / 'data' / 'flows.csv'
os.makedirs(TEMP_DIR, exist_ok=True)

FILE_REGISTRY = []
# non-attack: tráfico de control de la botnet (escaneo, infección y C&C), malicioso
for i in range(1, 31):
    FILE_REGISTRY.append((f'non-attack{i}.pcapng', 'Mirai_CnC', 1))
# non-botnet: tráfico limpio de la cámara IP, normal
for i in range(1, 31):
    if i == 16:
        continue
    FILE_REGISTRY.append((f'non-botnet{i}.pcapng', 'Normal', 0))
ATTACK_FILES = [
    ('ack_attack', 'ACK_Flood', 1),
    ('dns_attack', 'DNS_Flood', 1),
    ('greip_attack', 'GREIP_Flood', 1),
    ('http_attack', 'HTTP_Flood', 1),
    ('syn_attack', 'SYN_Flood', 1),
    ('udp_attack', 'UDP_Flood', 1),
    ('vse_attack', 'VSE_Flood', 1),
]
for base, class_name, label in ATTACK_FILES:
    for i in range(1, 4):
        FILE_REGISTRY.append((f'{base}{i}.pcapng', class_name, label))
print(f'{len(FILE_REGISTRY)} capturas')

80 capturas


In [2]:
PORT_TO_STATE = {53: 'DNS', 80: 'HTTP', 443: 'HTTPS', 22: 'SSH'}


def get_state(proto, dport):
    if proto == 6:
        return PORT_TO_STATE.get(dport, 'TCP_OTHER')
    elif proto == 17:
        return PORT_TO_STATE.get(dport, 'UDP_OTHER')
    else:
        return 'OTHER'


def convert_to_pcap(pcapng_path):
    pcap_path = TEMP_DIR / (Path(pcapng_path).stem + '.pcap')
    subprocess.run([EDITCAP, '-F', 'pcap', str(pcapng_path), str(pcap_path)],
                   capture_output=True, timeout=120, check=True)
    return pcap_path


def extract_flows(pcap_path, class_name, label):
    flows = defaultdict(lambda: {
        'timestamps': [], 'states': [],
        'n_pkts': 0, 'n_bytes': 0,
        'f_pkts': 0, 'f_bytes': 0,
        'b_pkts': 0, 'b_bytes': 0,
    })
    with open(pcap_path, 'rb') as f:
        reader = dpkt.pcap.Reader(f)
        for ts, raw in reader:
            try:
                eth = dpkt.ethernet.Ethernet(raw)
                if not isinstance(eth.data, dpkt.ip.IP):
                    continue
                ip = eth.data
                proto = ip.p
                pkt_len = len(raw)
                if isinstance(ip.data, (dpkt.tcp.TCP, dpkt.udp.UDP)):
                    sport = ip.data.sport
                    dport = ip.data.dport
                else:
                    sport = dport = 0
                fwd_key = (ip.src, ip.dst, sport, dport, proto)
                bwd_key = (ip.dst, ip.src, dport, sport, proto)
                if bwd_key in flows:
                    flows[bwd_key]['b_pkts'] += 1
                    flows[bwd_key]['b_bytes'] += pkt_len
                    flows[bwd_key]['n_pkts'] += 1
                    flows[bwd_key]['n_bytes'] += pkt_len
                else:
                    key = fwd_key
                    flows[key]['timestamps'].append(ts)
                    flows[key]['states'].append(get_state(proto, dport))
                    flows[key]['n_pkts'] += 1
                    flows[key]['n_bytes'] += pkt_len
                    flows[key]['f_pkts'] += 1
                    flows[key]['f_bytes'] += pkt_len
            except Exception:
                continue
    records = []
    for (src_ip, dst_ip, sport, dport, proto), d in flows.items():
        if not d['timestamps']:
            continue
        ts_arr = d['timestamps']
        duration = ts_arr[-1] - ts_arr[0] if len(ts_arr) > 1 else 0.0
        records.append({
            'src_ip': src_ip,
            'flow_start': ts_arr[0],
            'duration': duration,
            'state': d['states'][0],
            'n_pkts': d['n_pkts'],
            'n_bytes': d['n_bytes'],
            'f_pkts': d['f_pkts'],
            'f_bytes': d['f_bytes'],
            'b_pkts': d['b_pkts'],
            'b_bytes': d['b_bytes'],
            'avg_pkt_size': d['n_bytes'] / d['n_pkts'],
            'class_name': class_name,
            'label': label,
        })
    return records

In [3]:
all_records = []
for filename, class_name, label in FILE_REGISTRY:
    pcap_path = convert_to_pcap(DATA_PATH / filename)
    records = extract_flows(pcap_path, class_name, label)
    os.remove(pcap_path)
    all_records.extend(records)
    print(f'{filename:22s} {class_name:12s} {len(records):>8,} flujos')

df = pd.DataFrame(all_records)
df.to_csv(OUT_CSV, index=False)
print(f'\nShape: {df.shape}')
print(df['class_name'].value_counts().to_string())

non-attack1.pcapng     Mirai_CnC         781 flujos
non-attack2.pcapng     Mirai_CnC         698 flujos
non-attack3.pcapng     Mirai_CnC         717 flujos
non-attack4.pcapng     Mirai_CnC         716 flujos
non-attack5.pcapng     Mirai_CnC         868 flujos
non-attack6.pcapng     Mirai_CnC         873 flujos
non-attack7.pcapng     Mirai_CnC         859 flujos
non-attack8.pcapng     Mirai_CnC         712 flujos
non-attack9.pcapng     Mirai_CnC         832 flujos
non-attack10.pcapng    Mirai_CnC         846 flujos
non-attack11.pcapng    Mirai_CnC         841 flujos
non-attack12.pcapng    Mirai_CnC         847 flujos
non-attack13.pcapng    Mirai_CnC         784 flujos
non-attack14.pcapng    Mirai_CnC         805 flujos
non-attack15.pcapng    Mirai_CnC         809 flujos
non-attack16.pcapng    Mirai_CnC         918 flujos
non-attack17.pcapng    Mirai_CnC         899 flujos
non-attack18.pcapng    Mirai_CnC         929 flujos
non-attack19.pcapng    Mirai_CnC         879 flujos
non-attack20